# Prototype: does "local sensitivity model + one confirmation tile" converge?

**Supports one specific claim in the case study (section 4.3 and 5):** if we know roughly how each ink channel moves the fired colour (from a calibration chart), and we anchor to a tile we just measured, a bounded least-squares solve proposes a per-channel edit that gets the next tile inside tolerance in about one round, instead of repeated 6-7 h guess-and-check rounds.

**What this is NOT.** Everything below is **synthetic**. The "printer + kiln" is a made-up nonlinear function with hidden drift. It shows the *mechanism works if the device behaves like this*. It does not show the real tile process behaves like this. That is exactly what weeks 0-2 and the week-2 repeatability gate in the case study are for. Numbers here must not be quoted as expected client performance.

Assumptions baked into the simulation (all in `sim.py`):
- 6 ink channels, 7 measurement regions per tile, fired colour is a saturating, interacting function of ink amounts.
- Day-to-day drift is mostly **per-channel gain** (ink lot / kiln) plus a small uncorrectable Lab offset (0.25). This is the key assumption. Section 3 checks that a perfect oracle can fix it, otherwise the test would be meaningless.
- Calibration chart: 250 noisy patches. The fitted model is deliberately **wrong** about the true structure (it ignores per-channel saturation differences), so the confirmation tile has something to correct.
- Tolerance: dE00 mean <= 1.0 and max <= 1.5 over the 7 points (the case-study assumption).

In [1]:
import numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sim import *
rng = np.random.default_rng(7)
dev = Device(seed=3)
print("ready")

ready


## 1. Sanity check: is the scenario solvable at all?
If even an oracle (perfect knowledge of the hidden device) cannot reach tolerance, the test proves nothing. (My first version of this simulation failed this check, which is why the drift above is mostly gain.)

In [2]:
from scipy.optimize import least_squares
rows = []
for _ in range(40):
    E, master, gain, off = make_event(dev, rng)
    f = lambda d: DE(dev.lab(E * (1 + d), gain, off), master)
    sol = least_squares(f, np.zeros(C), bounds=(-.3, .3))
    rows.append((f(np.zeros(C)).mean(), f(sol.x).mean(), f(sol.x).max()))
rows = np.array(rows)
print(f"start: mean dE00 {rows[:,0].mean():.2f}  |  oracle best: mean {rows[:,1].mean():.2f}, max {rows[:,2].mean():.2f}")
print(f"share of events an oracle can bring inside tolerance: {np.mean((rows[:,1]<=1)&(rows[:,2]<=1.5)):.0%}")

start: mean dE00 1.67  |  oracle best: mean 0.19, max 0.30
share of events an oracle can bring inside tolerance: 100%


## 2. Calibrate the forward model from a chart, and check how wrong it is

In [3]:
X, Y = calibration_chart(dev, rng, n=250, sigma=0.3)
model = fit_model(X, Y)
Xt = np.vstack([tile_inks(rng) for _ in range(60)])
err = np.sqrt(((model(Xt) - dev.lab(Xt))**2).sum(1))
print(f"absolute Lab error of the fitted model on unseen colours: median {np.median(err):.2f}, p90 {np.percentile(err,90):.2f}")
print("(small, but not zero. The model is only trusted for the CHANGE from a measured tile, not for absolute colour.)")

absolute Lab error of the fitted model on unseen colours: median 0.29, p90 0.43
(small, but not zero. The model is only trusted for the CHANGE from a measured tile, not for absolute colour.)


## 3. One worked event, end to end
First sample fails, the advisor proposes per-channel % changes, one confirmation tile is measured.

In [4]:
rng2 = np.random.default_rng(21)
while True:                                   # pick a typical failing event (start mean dE00 between 1.5 and 2.6)
    E, master, gain, off = make_event(dev, rng2)
    p = print_measure(dev, E, gain, off, np.zeros(C), rng2, 0.3)
    if not passes(p, master) and 1.5 <= DE(p, master).mean() <= 2.6: break
delta = np.zeros(C)
print(f"First sample: mean dE00 {DE(p, master).mean():.2f}, max {DE(p, master).max():.2f}  -> fails tolerance")
for rnd in (1, 2, 3):
    step = advise(model, E * (1 + delta), p, master)
    delta = (1 + delta) * (1 + step) - 1
    p = print_measure(dev, E, gain, off, delta, rng2, 0.3)
    print()
    print(f"Round {rnd}: advisor proposes (relative change per ink): " + ", ".join(f"ink{i+1} {v*100:+.1f}%" for i, v in enumerate(step)))
    print(f"   confirmation tile: mean dE00 {DE(p, master).mean():.2f}, max {DE(p, master).max():.2f}  -> {'PASS, release batch' if passes(p, master) else 'not yet, next iteration'}")
    if passes(p, master): break

First sample: mean dE00 1.89, max 3.02  -> fails tolerance

Round 1: advisor proposes (relative change per ink): ink1 +14.8%, ink2 +6.2%, ink3 -7.3%, ink4 +12.3%, ink5 +20.7%, ink6 -10.3%
   confirmation tile: mean dE00 0.57, max 0.79  -> PASS, release batch


## 4. Monte Carlo: advisor vs a one-knob rule of thumb
The case study says the advisor must beat a naive rule, otherwise the extra machinery isn't justified. The naive rule scales *all* inks by one factor to cancel the lightness gap. Rounds = print + fire + measure cycles until inside tolerance (cap 4; a miss counts as 5).

In [5]:
N, SIG = 100, 0.3
rngm = np.random.default_rng(100)
adv = np.array([advisor_event(dev, model, rngm, SIG)[0] for _ in range(N)])
nai = np.array([naive_event(dev, model, rngm, SIG)[0] for _ in range(N)])
H_ROUND_ADV, H_ROUND_OLD = 1.6, 6.5          # hours per round: advisor loop vs today's assumed round
def summ(name, r):
    print(f"{name:<22} first-confirmation pass {np.mean(r==1):4.0%} | within 2 rounds {np.mean(r<=2):4.0%} | mean rounds {r.mean():.2f} | never converged {np.mean(r>4):.0%}")
summ("Advisor", adv); summ("One-knob rule of thumb", nai)
print()
print(f"Advisor: mean hours to approved file ~ {adv.mean()*H_ROUND_ADV:.1f} h (at {H_ROUND_ADV} h per round, an assumption)")
print("Case-study baseline for today's manual loop is an ASSUMED ~2.5 rounds x 6.5 h = ~16 h. It comes from the client's notes, not this simulation.")
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
bins = np.arange(0.5, 6.5, 1)
ax[0].hist([adv, nai], bins=bins, label=["Advisor", "One-knob rule"], color=["#1b7f5c", "#b0b0b0"])
ax[0].set_xlabel("rounds to pass (5 = did not converge)"); ax[0].set_ylabel("events"); ax[0].legend(); ax[0].set_title("Rounds to pass tolerance")
traces = np.array([advisor_trace(dev, model, rngm, SIG) for _ in range(50)])
ax[1].plot(traces.T, color="#1b7f5c", alpha=0.12); ax[1].plot(traces.mean(0), color="#1b7f5c", lw=3, label="mean")
ax[1].axhline(1.0, color="red", ls="--", label="tolerance (mean dE00 1.0)")
ax[1].set_xlabel("round (0 = first failed sample)"); ax[1].set_ylabel("mean dE00 vs master"); ax[1].set_title("Convergence per event"); ax[1].legend()
plt.tight_layout(); plt.savefig("convergence.png", dpi=130); plt.show()

Advisor                first-confirmation pass  94% | within 2 rounds 100% | mean rounds 1.06 | never converged 0%
One-knob rule of thumb first-confirmation pass  21% | within 2 rounds  23% | mean rounds 4.02 | never converged 72%

Advisor: mean hours to approved file ~ 1.7 h (at 1.6 h per round, an assumption)
Case-study baseline for today's manual loop is an ASSUMED ~2.5 rounds x 6.5 h = ~16 h. It comes from the client's notes, not this simulation.


C:\Users\ASUS\AppData\Local\Temp\ipykernel_5436\2397958579.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig("convergence.png", dpi=130); plt.show()


## 5. The week-2 gate: what happens when same-file print repeatability is bad?
This is the case study's kill-risk. `sigma_rep` is how much the shade moves when you print and fire the **same file** twice. The advisor can only converge if that noise is well under the tolerance.

In [6]:
sigs = [0.1, 0.2, 0.3, 0.5, 0.8, 1.2]
res = []
for s in sigs:
    r = np.array([advisor_event(dev, model, np.random.default_rng(1000 + i), s)[0] for i in range(40)])
    res.append((s, np.mean(r == 1), np.mean(r <= 2), r.mean()))
print("sigma_rep   first-pass   within 2 rounds   mean rounds")
for s, a, b, c in res: print(f"  {s:4.1f}       {a:5.0%}         {b:5.0%}           {c:.2f}")
plt.figure(figsize=(5.5, 3.4))
plt.plot([r[0] for r in res], [r[1] for r in res], "o-", label="first-confirmation pass")
plt.plot([r[0] for r in res], [r[2] for r in res], "s-", label="pass within 2 rounds")
plt.axvline(0.5, color="red", ls="--", label="week-2 stop line (0.5)"); plt.axhline(0.75, color="grey", ls=":", label="75% target")
plt.xlabel("same-file print+fire repeatability (sigma, dE-units)"); plt.ylabel("share of events"); plt.legend(fontsize=8); plt.tight_layout()
plt.savefig("repeatability_gate.png", dpi=130); plt.show()

sigma_rep   first-pass   within 2 rounds   mean rounds
   0.1        100%          100%           1.00
   0.2         98%          100%           1.02
   0.3         92%          100%           1.07
   0.5         57%           78%           1.82
   0.8          0%            5%           4.50
   1.2          0%            0%           5.00


C:\Users\ASUS\AppData\Local\Temp\ipykernel_5436\33186725.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.savefig("repeatability_gate.png", dpi=130); plt.show()


## 6. What this does and does not show

**Shows (given the synthetic device):**
- A calibration chart plus a local sensitivity solve can propose a per-channel edit that converges in about one confirmation round, even when the fitted model is imperfect, because the confirmation tile re-anchors it.
- A one-knob rule of thumb is clearly worse, so the per-channel solve earns its keep.
- Convergence degrades as same-file repeatability gets worse, and the week-2 stop line in the case study sits where this starts to hurt. That is the reason for testing it first.

**Does not show:**
- Anything about the real printer, kiln, inks or tiles. Drift being mostly per-channel gain is my assumption. If real drift is mostly non-correctable (kiln position, body colour), the oracle check in section 1 would fail and the plan changes.
- Patterned-tile registration, instrument error beyond a flat noise term, gloss, or metamerism.
- How today's QC team actually performs. I did not simulate a manual baseline on purpose: a made-up one would just reflect my assumptions. The case study's baseline must come from the client's records.